# Bài 1.4 - Sửa kernel Triton của notebook 05

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KuSenpai/modelopt101/blob/exercise-1.3-1.4/exercises/1.4_triton_kernels.ipynb)

Kernel matmul của `examples/05_kernel_optimization.ipynb` dùng một cấu hình cố định
(`BLOCK 64×64×32`, `num_warps`/`num_stages` mặc định). Lần chạy trước trên T4 nó chỉ đạt khoảng
**1 TFLOP/s, 0.06x `torch.matmul`**. Bài này sửa kernel đó cho gần cuBLAS:

1. **`@triton.autotune`** trên nhiều `BLOCK_M/N/K`, `num_warps`, `num_stages`; input FP16, cộng dồn FP32.
2. **Fuse ReLU** vào epilogue của matmul, so với `torch.relu(torch.matmul(...))`.
3. **GROUP_M = 1 so với GROUP_M = 8**: cùng tile config, chỉ đổi thứ tự duyệt tile, để đo riêng hiệu ứng L2 cache.
4. Theo tutorial chính thức của Triton: **fused softmax** và **layer norm** (forward), so với torch.
5. Benchmark nhiều kích thước bằng `triton.testing.do_bench` (median), báo TFLOP/s và **% so với cuBLAS**.

**Chỉ tiêu:** kernel matmul đạt **≥ 70% throughput cuBLAS** (`torch.matmul` FP16).

**Runtime:** `Runtime -> Change runtime type` -> chọn **L4 hoặc A100** nếu có (Ampere trở lên: có `cp.async`
nên Triton pipeline được việc nạp tile, và tensor core mạnh hơn). Không có thì chọn **T4** (Turing, sm_75).
Khoảng 10 phút, phần lớn là autotune.

## 1. Setup

In [ ]:
import numpy as np
import torch
import triton
import triton.language as tl
import matplotlib.pyplot as plt

torch.manual_seed(0)
assert torch.cuda.is_available(), "Set Runtime -> Change runtime type -> GPU"
device = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
GPU = props.name
CC = torch.cuda.get_device_capability()
print(f"GPU: {GPU} | sm_{CC[0]}{CC[1]} | {props.multi_processor_count} SMs | "
      f"L2 {props.L2_cache_size / 2**20:.0f} MB | {props.total_memory / 2**30:.0f} GB")
print(f"torch {torch.__version__} | triton {triton.__version__} | CUDA {torch.version.cuda}")
if CC < (8, 0):
    print("pre-Ampere GPU: no cp.async, so num_stages > 2 gives little; expect a lower % of cuBLAS")

DTYPE = torch.float16

def bench(fn):
    """Median runtime in ms (do_bench flushes L2 between runs and handles warmup)."""
    return triton.testing.do_bench(fn, warmup=25, rep=100, return_mode="median")

## 2. Kernel matmul: autotune, FP16 → FP32 accumulate, fused ReLU, GROUP_M

Giữ nguyên cấu trúc kernel của notebook 05 (tiling theo K, tile `BLOCK_M × BLOCK_N` cho mỗi program,
thứ tự tile theo nhóm `GROUP_M`), và sửa bốn chỗ:

- **Accumulator FP32**, `tl.dot(a, b, acc)` để tensor core cộng thẳng vào accumulator, chỉ ép về FP16 khi ghi ra.
- **`ACTIVATION` là `tl.constexpr`**: khi bằng `"relu"`, `tl.maximum(acc, 0)` chạy trên thanh ghi trước khi ghi
  ra bộ nhớ, nên ReLU gần như miễn phí (không có thêm một lượt đọc/ghi `M×N` như `torch.relu` riêng).
- **Sửa công thức `pid_m`**: notebook 05 dùng `pid % group_size_m`, chỉ đúng khi nhóm cuối đủ `GROUP_M`
  hàng tile. Tutorial dùng `(pid % num_pid_in_group) % group_size_m`.
- **Autotune**: `matmul_kernel` gốc là một `@triton.jit`; `triton.autotune(...)(matmul_kernel)` bọc nó bằng
  danh sách config. Lần gọi đầu với mỗi `(M, N, K)`, Triton chạy thử mọi config và nhớ config nhanh nhất.
  Config nào dùng quá shared memory của GPU thì bị bỏ qua tự động.

`GROUP_M` và `ACTIVATION` **không** nằm trong config autotune mà truyền khi gọi: nhờ vậy khi so `GROUP_M = 1`
với `GROUP_M = 8`, hai bên dùng đúng cùng một tile config.

In [ ]:
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr, M, N, K,
    stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
    GROUP_M: tl.constexpr, ACTIVATION: tl.constexpr,
):
    # grouped tile order: GROUP_M rows of tiles are walked column by column, so the
    # A rows and B columns they share are still in L2 when the next tile needs them
    pid = tl.program_id(0)
    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    num_pid_in_group = GROUP_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    offs_am = (pid_m * BLOCK_M + tl.arange(0, BLOCK_M)) % M
    offs_bn = (pid_n * BLOCK_N + tl.arange(0, BLOCK_N)) % N
    offs_k = tl.arange(0, BLOCK_K)
    a_ptrs = a_ptr + offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak
    b_ptrs = b_ptr + offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)       # FP32 accumulator
    for k in range(0, tl.cdiv(K, BLOCK_K)):
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_K, other=0.0)
        acc = tl.dot(a, b, acc)                                  # fp16 x fp16 -> fp32 on tensor cores
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    if ACTIVATION == "relu":                                     # fused epilogue, still in registers
        acc = tl.maximum(acc, 0.0)
    c = acc.to(c_ptr.dtype.element_ty)

    offs_cm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_cn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    tl.store(c_ptrs, c, mask=(offs_cm[:, None] < M) & (offs_cn[None, :] < N))


def cfg(bm, bn, bk, warps, stages):
    return triton.Config({"BLOCK_M": bm, "BLOCK_N": bn, "BLOCK_K": bk}, num_warps=warps, num_stages=stages)

# the configs of the official Triton matmul tutorial, plus smaller-stage variants that fit
# the 64 KB of shared memory per block on Turing (T4)
CONFIGS = [
    cfg(128, 256, 64, 8, 3), cfg(64, 256, 32, 4, 4), cfg(128, 128, 32, 4, 4), cfg(128, 64, 32, 4, 4),
    cfg(64, 128, 32, 4, 4),  cfg(128, 32, 32, 4, 4), cfg(64, 32, 32, 2, 5),   cfg(32, 64, 32, 2, 5),
    cfg(128, 256, 32, 8, 2), cfg(256, 128, 32, 8, 2), cfg(128, 128, 32, 8, 2), cfg(128, 128, 64, 8, 2),
    cfg(128, 128, 64, 4, 2), cfg(64, 128, 64, 4, 2),  cfg(128, 64, 64, 4, 2),  cfg(64, 64, 64, 4, 2),
]
matmul_tuned = triton.autotune(configs=CONFIGS, key=["M", "N", "K"])(matmul_kernel)


def _launch(kernel, a, b, activation, group_m, **meta):
    assert a.shape[1] == b.shape[0] and a.dtype == b.dtype == DTYPE
    M, K = a.shape
    N = b.shape[1]
    c = torch.empty((M, N), device=a.device, dtype=DTYPE)
    grid = lambda META: (triton.cdiv(M, META["BLOCK_M"]) * triton.cdiv(N, META["BLOCK_N"]),)
    kernel[grid](a, b, c, M, N, K, a.stride(0), a.stride(1), b.stride(0), b.stride(1),
                 c.stride(0), c.stride(1), GROUP_M=group_m, ACTIVATION=activation, **meta)
    return c

def matmul(a, b, activation="", group_m=8):
    """Autotuned Triton matmul (C = act(A @ B))."""
    return _launch(matmul_tuned, a, b, activation, group_m)

def matmul_nb05(a, b):
    """The fixed configuration notebook 05 used: 64x64x32 tiles, default warps / stages."""
    return _launch(matmul_kernel, a, b, "", 8, BLOCK_M=64, BLOCK_N=64, BLOCK_K=32)

## 3. Kiểm tra đúng

So với `torch.matmul` trên vài kích thước, kể cả kích thước lẻ (không chia hết cho tile) để thử phần mask.
Output là FP16 nên sai khác cỡ 1 ULP của FP16 ở độ lớn của kết quả là bình thường.

In [ ]:
for (M, N, K) in [(512, 512, 512), (513, 1000, 777), (64, 4096, 1024), (2048, 96, 300)]:
    a = torch.randn(M, K, device=device, dtype=DTYPE)
    b = torch.randn(K, N, device=device, dtype=DTYPE)
    ref = torch.matmul(a, b)
    for name, out, want in (("matmul", matmul(a, b), ref),
                            ("matmul+relu", matmul(a, b, "relu"), torch.relu(ref)),
                            ("GROUP_M=1", matmul(a, b, group_m=1), ref),
                            ("nb05 config", matmul_nb05(a, b), ref)):
        err = (out.float() - want.float()).abs().max().item()
        ok = torch.allclose(out, want, atol=1e-2, rtol=1e-2)
        print(f"{str((M, N, K)):<20}{name:<13} max|diff| = {err:.4f}  {'OK' if ok else 'MISMATCH'}")
        assert ok

## 4. Benchmark matmul: TFLOP/s và % cuBLAS

Ma trận vuông từ 256 đến 8192, cộng hai hình chữ nhật kiểu lớp MLP của LLM (4096 token, hidden 4096 ↔ 11008).
Mỗi kích thước đo bằng `do_bench` (median), `TFLOP/s = 2·M·N·K / thời gian`. Các cột:

- **cuBLAS**: `torch.matmul` FP16;
- **Triton tuned**: kernel autotune ở trên (GROUP_M = 8), kèm config autotune chọn;
- **nb05 cfg**: cùng kernel nhưng với config cố định của notebook 05, để thấy riêng tác dụng của autotune.

In [ ]:
SHAPES = [(s, s, s) for s in (256, 512, 1024, 2048, 3072, 4096, 6144, 8192)] + \
         [(4096, 11008, 4096), (4096, 4096, 11008)]

def short_cfg(c):
    k = c.kwargs
    return f"{k['BLOCK_M']}x{k['BLOCK_N']}x{k['BLOCK_K']} w{c.num_warps} s{c.num_stages}"

rows = []
for (M, N, K) in SHAPES:
    a = torch.randn(M, K, device=device, dtype=DTYPE)
    b = torch.randn(K, N, device=device, dtype=DTYPE)
    tf = lambda ms: 2 * M * N * K / ms / 1e9
    r = {"shape": (M, N, K)}
    r["cublas"] = tf(bench(lambda: torch.matmul(a, b)))
    r["triton"] = tf(bench(lambda: matmul(a, b)))          # first call autotunes this shape
    r["config"] = short_cfg(matmul_tuned.best_config)
    r["nb05"]   = tf(bench(lambda: matmul_nb05(a, b)))
    r["torch_relu"]  = bench(lambda: torch.relu(torch.matmul(a, b)))
    r["triton_relu"] = bench(lambda: matmul(a, b, "relu"))
    r["g1"] = bench(lambda: matmul(a, b, group_m=1))         # same tile config, row-major tile order
    r["g8"] = bench(lambda: matmul(a, b, group_m=8))
    rows.append(r)
    del a, b

print(f"GPU: {GPU}\n")
print(f"{'M x N x K':<20}{'cuBLAS':>8}{'Triton':>8}{'% cuBLAS':>10}{'nb05 cfg':>10}{'% cuBLAS':>10}   autotuned config")
print(f"{'':<20}{'TF/s':>8}{'TF/s':>8}{'':>10}{'TF/s':>10}")
print("-" * 98)
for r in rows:
    print(f"{'x'.join(map(str, r['shape'])):<20}{r['cublas']:>8.2f}{r['triton']:>8.2f}"
          f"{100 * r['triton'] / r['cublas']:>9.1f}%{r['nb05']:>10.2f}{100 * r['nb05'] / r['cublas']:>9.1f}%   {r['config']}")

big = [r for r in rows if min(r["shape"]) >= 1024]
pct_big = [100 * r["triton"] / r["cublas"] for r in big]
mean_pct = float(np.mean(pct_big))
print(f"\nmean % of cuBLAS over the {len(big)} shapes with min(M,N,K) >= 1024: {mean_pct:.1f}%  "
      f"(min {min(pct_big):.1f}%, {sum(p >= 70 for p in pct_big)}/{len(big)} shapes >= 70%)")
print(f"target >= 70% of cuBLAS -> {'PASS' if mean_pct >= 70 else 'FAIL'}")

In [ ]:
labels = ["x".join(map(str, r["shape"])) if len(set(r["shape"])) > 1 else str(r["shape"][0]) for r in rows]
x = np.arange(len(rows))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(x, [r["cublas"] for r in rows], "o-", color="#555", label="cuBLAS (torch.matmul)")
ax[0].plot(x, [r["triton"] for r in rows], "o-", color="#5cb85c", label="Triton autotuned")
ax[0].plot(x, [r["nb05"] for r in rows], "o--", color="#d9534f", label="Triton, nb05 config")
ax[0].set_ylabel("TFLOP/s"); ax[0].set_title(f"FP16 matmul throughput on {GPU}")
ax[0].set_xticks(x, labels, rotation=30, fontsize=8); ax[0].grid(alpha=0.3); ax[0].legend()
pct = [100 * r["triton"] / r["cublas"] for r in rows]
bars = ax[1].bar(x, pct, color=["#5cb85c" if p >= 70 else "#f0ad4e" for p in pct])
ax[1].bar_label(bars, fmt="%.0f%%", fontsize=8)
ax[1].axhline(70, color="#d9534f", ls="--", lw=1, label="target 70%")
ax[1].set_ylabel("% of cuBLAS"); ax[1].set_title("Triton autotuned / cuBLAS")
ax[1].set_xticks(x, labels, rotation=30, fontsize=8); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Fused ReLU và hiệu ứng GROUP_M (L2 cache)

**Fused ReLU.** `torch.relu(torch.matmul(a, b))` chạy hai kernel: matmul ghi `C` ra bộ nhớ, rồi ReLU đọc lại
`C` và ghi lần nữa. Bản Triton làm ReLU trên thanh ghi nên bỏ được lượt đọc/ghi `2·M·N·2` byte đó.

**GROUP_M.** Với `GROUP_M = 1`, các program được xếp theo hàng: một hàng tile của `C` đọc cùng một dải hàng của `A`
nhưng phải đọc **toàn bộ** `B`; khi `B` lớn hơn L2 thì mỗi hàng tile đọc lại `B` từ DRAM. Với `GROUP_M = 8`,
các program chạy cùng lúc phủ một khối gần vuông `8 × n` tile, nên các cột của `B` vừa nạp vào L2 được nhiều
hàng tile dùng lại. Cùng tile config, cùng số FLOP, chỉ khác thứ tự, nên chênh lệch thời gian là hiệu ứng L2.

In [ ]:
print(f"{'M x N x K':<20}{'torch mm+relu':>14}{'Triton fused':>14}{'speedup':>9}"
      f"{'GROUP_M=1':>12}{'GROUP_M=8':>12}{'G8 vs G1':>10}")
print(f"{'':<20}{'ms':>14}{'ms':>14}{'':>9}{'TF/s':>12}{'TF/s':>12}")
print("-" * 91)
for r in rows:
    M, N, K = r["shape"]
    tf = lambda ms: 2 * M * N * K / ms / 1e9
    print(f"{'x'.join(map(str, r['shape'])):<20}{r['torch_relu']:>14.3f}{r['triton_relu']:>14.3f}"
          f"{r['torch_relu'] / r['triton_relu']:>8.2f}x{tf(r['g1']):>12.2f}{tf(r['g8']):>12.2f}"
          f"{r['g1'] / r['g8']:>9.2f}x")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].bar(x - 0.2, [r["torch_relu"] / r["triton_relu"] for r in rows], 0.4, color="#5b8def",
          label="torch.relu(torch.matmul) / Triton fused")
ax[0].bar(x + 0.2, [(r["torch_relu"] / r["triton_relu"]) * (r["triton"] / r["cublas"]) for r in rows], 0.4,
          color="#ccc", label="same, normalised by unfused Triton/cuBLAS")
ax[0].axhline(1, color="gray", lw=1); ax[0].set_title("fused ReLU: speedup over torch (>1 = Triton faster)")
ax[0].set_xticks(x, labels, rotation=30, fontsize=8); ax[0].legend(fontsize=8)
ax[1].bar(x, [r["g1"] / r["g8"] for r in rows], color="#f0ad4e")
ax[1].axhline(1, color="gray", lw=1)
ax[1].set_title("GROUP_M=8 speedup over GROUP_M=1 (same tile config)")
ax[1].set_xticks(x, labels, rotation=30, fontsize=8)
plt.tight_layout(); plt.show()

## 6. Fused softmax (theo tutorial Triton)

Softmax theo hàng viết bằng các op của torch (`x.max`, `x - max`, `exp`, `sum`, `/`) đọc và ghi lại tensor nhiều lần.
Kernel fused đọc mỗi hàng một lần vào thanh ghi/SRAM, tính max, exp, tổng và chia tại chỗ, rồi ghi một lần:
lý tưởng là `2 · numel · 4` byte (FP32). Một program xử lý một hàng, `BLOCK_SIZE` = lũy thừa 2 ≥ số cột.
Softmax là phép **memory-bound**, nên đơn vị là **GB/s** chứ không phải TFLOP/s.

In [ ]:
@triton.jit
def softmax_kernel(out_ptr, in_ptr, in_row_stride, out_row_stride, n_cols, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    offs = tl.arange(0, BLOCK_SIZE)
    mask = offs < n_cols
    x = tl.load(in_ptr + row * in_row_stride + offs, mask=mask, other=-float("inf"))
    x = x - tl.max(x, axis=0)              # subtract the row max for numerical stability
    num = tl.exp(x)
    tl.store(out_ptr + row * out_row_stride + offs, num / tl.sum(num, axis=0), mask=mask)

def softmax(x):
    n_rows, n_cols = x.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    num_warps = 4 if BLOCK_SIZE < 2048 else 8 if BLOCK_SIZE < 4096 else 16
    y = torch.empty_like(x)
    softmax_kernel[(n_rows,)](y, x, x.stride(0), y.stride(0), n_cols, BLOCK_SIZE=BLOCK_SIZE, num_warps=num_warps)
    return y

def naive_softmax(x):
    """Row softmax from separate torch ops: ~5 reads + 3 writes of the tensor."""
    x_max = x.max(dim=1)[0]
    z = x - x_max[:, None]
    num = torch.exp(z)
    return num / num.sum(dim=1)[:, None]

x = torch.randn(1823, 781, device=device)
assert torch.allclose(softmax(x), torch.softmax(x, dim=1), atol=1e-6), "softmax mismatch"
print("softmax matches torch.softmax")

SM_M = 4096
sm_rows = []
for n in (256, 512, 1024, 2048, 4096, 8192, 12288):
    x = torch.randn(SM_M, n, device=device)
    gbs = lambda ms: 2 * x.numel() * x.element_size() / ms / 1e6
    sm_rows.append((n, gbs(bench(lambda: softmax(x))), gbs(bench(lambda: torch.softmax(x, dim=1))),
                    gbs(bench(lambda: naive_softmax(x)))))

print(f"\nsoftmax, {SM_M} rows, fp32 (GB/s, counting one read + one write)")
print(f"{'cols':>7}{'Triton':>9}{'torch':>9}{'naive':>9}{'Triton/torch':>14}")
for n, t, s, nv in sm_rows:
    print(f"{n:>7}{t:>9.0f}{s:>9.0f}{nv:>9.0f}{t / s:>13.2f}x")

## 7. Layer norm forward (theo tutorial Triton)

Mỗi program chuẩn hoá một hàng: lượt 1 cộng dồn để lấy mean, lượt 2 lấy phương sai, lượt 3 ghi
`(x − mean) · rstd · w + b`. Cộng dồn trong FP32 dù input là FP16. Khi hàng vừa một block (`N ≤ BLOCK_SIZE`),
ba lượt đọc cùng một vùng nhớ nên hai lượt sau đi từ L1/L2. Chỉ làm forward; tutorial có thêm backward.

In [ ]:
@triton.jit
def layer_norm_fwd(X, Y, W, B, stride, N, eps, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    X += row * stride
    Y += row * stride
    _mean = tl.zeros([BLOCK_SIZE], dtype=tl.float32)
    for off in range(0, N, BLOCK_SIZE):
        cols = off + tl.arange(0, BLOCK_SIZE)
        _mean += tl.load(X + cols, mask=cols < N, other=0.0).to(tl.float32)
    mean = tl.sum(_mean, axis=0) / N
    _var = tl.zeros([BLOCK_SIZE], dtype=tl.float32)
    for off in range(0, N, BLOCK_SIZE):
        cols = off + tl.arange(0, BLOCK_SIZE)
        x = tl.load(X + cols, mask=cols < N, other=0.0).to(tl.float32)
        x = tl.where(cols < N, x - mean, 0.0)
        _var += x * x
    rstd = 1 / tl.sqrt(tl.sum(_var, axis=0) / N + eps)
    for off in range(0, N, BLOCK_SIZE):
        cols = off + tl.arange(0, BLOCK_SIZE)
        mask = cols < N
        w = tl.load(W + cols, mask=mask)
        b = tl.load(B + cols, mask=mask)
        x = tl.load(X + cols, mask=mask, other=0.0).to(tl.float32)
        tl.store(Y + cols, ((x - mean) * rstd * w + b).to(Y.dtype.element_ty), mask=mask)

def layer_norm(x, w, b, eps=1e-5):
    M, N = x.shape
    BLOCK_SIZE = min(65536 // x.element_size(), triton.next_power_of_2(N))
    num_warps = min(max(BLOCK_SIZE // 256, 1), 8)
    y = torch.empty_like(x)
    layer_norm_fwd[(M,)](x, y, w, b, x.stride(0), N, eps, BLOCK_SIZE=BLOCK_SIZE, num_warps=num_warps)
    return y

x = torch.randn(1151, 8192 + 13, device=device, dtype=DTYPE)
w = torch.rand(x.shape[1], device=device, dtype=DTYPE)
b = torch.randn(x.shape[1], device=device, dtype=DTYPE)
ref = torch.nn.functional.layer_norm(x, (x.shape[1],), w, b)
assert torch.allclose(layer_norm(x, w, b), ref, atol=1e-2, rtol=0), "layer norm mismatch"
print("layer norm matches torch.nn.functional.layer_norm")

LN_M = 4096
ln_rows = []
for n in (512, 1024, 2048, 4096, 8192, 16384):
    x = torch.randn(LN_M, n, device=device, dtype=DTYPE)
    w = torch.rand(n, device=device, dtype=DTYPE)
    b = torch.randn(n, device=device, dtype=DTYPE)
    gbs = lambda ms: 2 * x.numel() * x.element_size() / ms / 1e6
    ln_rows.append((n, gbs(bench(lambda: layer_norm(x, w, b))),
                    gbs(bench(lambda: torch.nn.functional.layer_norm(x, (n,), w, b)))))

print(f"\nlayer norm forward, {LN_M} rows, fp16 (GB/s, counting one read + one write of x)")
print(f"{'cols':>7}{'Triton':>9}{'torch':>9}{'Triton/torch':>14}")
for n, t, s in ln_rows:
    print(f"{n:>7}{t:>9.0f}{s:>9.0f}{t / s:>13.2f}x")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
c = [r[0] for r in sm_rows]
ax[0].plot(c, [r[1] for r in sm_rows], "o-", color="#5cb85c", label="Triton fused")
ax[0].plot(c, [r[2] for r in sm_rows], "o-", color="#555", label="torch.softmax")
ax[0].plot(c, [r[3] for r in sm_rows], "o--", color="#d9534f", label="naive torch ops")
ax[0].set_xscale("log", base=2); ax[0].set_xlabel("columns"); ax[0].set_ylabel("GB/s")
ax[0].set_title(f"softmax ({SM_M} rows, fp32)"); ax[0].legend(); ax[0].grid(alpha=0.3)
c = [r[0] for r in ln_rows]
ax[1].plot(c, [r[1] for r in ln_rows], "o-", color="#5cb85c", label="Triton")
ax[1].plot(c, [r[2] for r in ln_rows], "o-", color="#555", label="torch layer_norm")
ax[1].set_xscale("log", base=2); ax[1].set_xlabel("columns"); ax[1].set_ylabel("GB/s")
ax[1].set_title(f"layer norm forward ({LN_M} rows, fp16)"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.suptitle(GPU); plt.tight_layout(); plt.show()

## 8. Nhận xét

*(điền sau khi chạy trên Colab)*